Installing the dependencies - langchain framework and flan-T5 model

In [ ]:
!pip install -q --upgrade \
langchain \
langchain-community \
transformers \
sentence-transformers \
faiss-cpu \
pypdf \
requests==2.32.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 93.4 MB/s eta 0:00:00


In [ ]:
#Imports

from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import pipeline

In [ ]:
from langchain.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/KMS Company Profile.pdf")
documents = loader.load()

text_splitter = CharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

Embedding with hugging face model

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Store vector in FAISS Database

In [ ]:
db = FAISS.from_documents(docs, embeddings)
retriever = db.as_retriever()

LLM- Free

In [ ]:
pipe = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_length=200
)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM',

RAG

In [ ]:
def ask_bot(question):
    relevant_docs = retriever.get_relevant_documents(question)

    context = " ".join([doc.page_content for doc in relevant_docs])

    prompt = f"""
    Answer ONLY from the context below.
    If the answer is not in the context, say "I don't know".

    Context:
    {context}

    Question: {question}
    """

    result = pipe(prompt)
    return result[0]['generated_text']

Testing

In [ ]:
print(ask_bot("What does Kect Management Services do?"))
print(ask_bot("What documents are required?"))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (4397 > 512). Running this sequence through the model will result in indexing errors



    Answer ONLY from the context below.
    If the answer is not in the context, say "I don't know".

    Context:
    KECT MANAGEMENT 
SERVICES (PVT) LTD
/gid00007/gid00042/gid00045/gid00032/gid00036/gid00034/gid00041/gid00001/gid00019/gid00032/gid00030/gid00045/gid00048/gid00036/gid00047/gid00040/gid00032/gid00041/gid00047/gid00001/gid00020/gid00032/gid00045/gid00049/gid00036/gid00030/gid00032/gid00046
/gid00004/gid00016/gid00014/gid00017/gid00002/gid00015/gid00026/gid00001/gid00017/gid00019/gid00016/gid00007/gid00010/gid00013/gid00006
Registered Office : 110/1A, Cotta Road, Colombo 08
Email : kms@kectholdings.com   Web : www.kectkms.com
Tel : +94 11 268 6554
Mob : +94 77296 9370 - Mr. V .K. Khandelwal (Chairman) 
           +94 70 536 3099 - Ms. Chathuranganie Amarathunga (Director) /gid00011/gid00042/gid00029/gid00001/gid00004/gid00028/gid00047/gid00032/gid00034/gid00042/gid00045/gid00036/gid00032/gid00046
1.   Construction Workers 
 2.    Hospital
4.    Auxiliary Police Officers


KeyboardInterrupt: 

In [ ]:
while True:
    q = input("\nAsk something (or type exit): ")
    if q.lower() == "exit":
        break
    print("Bot:", ask_bot(q))

Bot: 
    Answer ONLY from the context below.
    If the answer is not in the context, say "I don't know".

    Context:
    /gid00011/gid00042/gid00029/gid00001/gid00004/gid00028/gid00047/gid00032/gid00034/gid00042/gid00045/gid00036/gid00032/gid00046
1.   Construction Workers 
 2.    Hospital
4.    Auxiliary Police Officers
       Security Guards and Other
       Security Personnel
3.    Hotel
Page 06 /gid00022/gid00002/gid00006/gid00001/gid00011/gid00042/gid00029/gid00001/gid00016/gid00045/gid00031/gid00032/gid00045/gid00046
Requirements from Sri Lanka
Open Positions :
1. 6G Welder - 10 Nos.
2. Carpenter - 10 Nos.
3. Driver - 10 Nos.
4. Electrician - 10 Nos.
5. Furniture Carpenter - 10 Nos.
6. Mason - 10 Nos.
7. Plumber - 10 Nos.
8. Scaﬀ Folder - 10 Nos.
9. Security Guards - 10 Nos.
10.
11.
  
/gid00024/gid00032/gid00001/gid00035/gid00028/gid00049/gid00032/gid00001/gid00028/gid00039/gid00046/gid00042/gid00001/gid00045/gid00032/gid00030/gid00032/gid00036/gid00049/gid00032/gid00031/gid